In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns',None)

In [2]:
df = pd.read_csv('data/raw/train.csv')

In [3]:
df= df.drop(df[df['Id'] == 1299].index)
df = df.drop(df[df['Id'] == 524].index)
df = df.dropna(subset=['Electrical'])

In [4]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(
    df.drop('SalePrice' , axis=1),
    df['SalePrice'],
    test_size=0.2,
    random_state=42,    
)

In [5]:
from sklearn.base import TransformerMixin , BaseEstimator # use for custom functoins
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , OrdinalEncoder , OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline 
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor

In [6]:
# custom transformer for dropping columns with nans and outliers
class DropFeatures(BaseEstimator , TransformerMixin):

    def __init__(self , features_to_drop = None , nan_threshold = 1):

        self.features_to_drop = features_to_drop
        self.nan_threshold = nan_threshold

    def fit(self , X , y = None):

        if self.features_to_drop:
            self.features_to_drop = self.features_to_drop

        else:
            nan_counts = X.isnull().sum()

            self.features_to_drop = list(nan_counts[nan_counts > self.nan_threshold].index) 

        return self
    
    def transform(self, X):

        X_transformed = X.copy()

        X_transformed = X_transformed.drop(columns = self.features_to_drop, axis = 1  , errors = 'ignore')
        # drop the rows with na
        return X_transformed    

In [7]:
testing_set = DropFeatures().fit_transform(X_train)
testing_set

,Id,MSSubClass,MSZoning,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,ExterQual,ExterCond,Foundation,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,GarageCars,GarageArea,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition
254,255,20,RL,8400,Pave,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,1Fam,1Story,5,6,1957,1957,Gable,CompShg,MetalSd,MetalSd,TA,Gd,CBlock,922,0,392,1314,GasA,TA,Y,SBrkr,1314,0,0,1314,1,0,1,0,3,1,TA,5,Typ,0,1,294,Y,250,0,0,0,0,0,0,6,2010,WD,Normal
1364,1365,160,FV,3180,Pave,Reg,Lvl,AllPub,Inside,Gtl,Somerst,Norm,Norm,TwnhsE,2Story,7,5,2005,2005,Gable,CompShg,MetalSd,MetalSd,Gd,TA,PConc,0,0,600,600,GasA,Ex,Y,SBrkr,520,600,80,1200,0,0,2,1,2,1,Gd,4,Typ,0,2,480,Y,0,166,0,0,0,0,0,4,2006,WD,Abnorml
637,638,190,RM,6000,Pave,Reg,Lvl,AllPub,Inside,Gtl,OldTown,Norm,Norm,2fmCon,1.5Fin,5,4,1954,1954,Gable,CompShg,Wd Sdng,Wd Sdng,TA,TA,CBlock,0,0,811,811,GasA,TA,Y,FuseA,811,576,0,1387,0,0,2,0,3,2,Gd,7,Typ,0,1,256,Y,0,0,0,0,0,0,0,11,2009,WD,Normal
974,975,70,RL,11414,Pave,IR1,Lvl,AllPub,Corner,Gtl,BrkSide,RRAn,Feedr,1Fam,2Story,7,8,1910,1993,Gable,CompShg,HdBoard,HdBoard,TA,Gd,BrkTil,0,0,728,728,GasA,TA,N,SBrkr,1136,883,0,2019,0,0,1,0,3,1,Gd,8,Typ,0,2,532,Y,509,135,0,0,0,0,0,10,2009,WD,Normal
514,515,45,RL,10594,Pave,Reg,Lvl,AllPub,Inside,Gtl,Crawfor,Norm,Norm,1Fam,1.5Unf,5,5,1926,1950,Gable,CompShg,Wd Sdng,Wd Sdng,TA,TA,BrkTil,0,0,768,768,Grav,Fa,N,SBrkr,789,0,0,789,0,0,1,0,2,1,TA,5,Typ,0,1,200,Y,0,0,112,0,0,0,0,6,2007,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1096,1097,70,RM,6882,Pave,Reg,Lvl,AllPub,Inside,Gtl,IDOTRR,Norm,Norm,1Fam,2Story,6,7,1914,2006,Gable,CompShg,Wd Sdng,Wd Sdng,TA,TA,PConc,0,0,684,684,GasA,TA,Y,SBrkr,773,582,0,1355,0,0,1,1,3,1,Gd,7,Typ,0,0,0,Y,136,0,115,0,0,0,0,3,2007,WD,Normal
1131,1132,20,RL,10712,Pave,Reg,Lvl,AllPub,Inside,Gtl,Mitchel,Norm,Norm,1Fam,1Story,5,5,1991,1992,Gable,CompShg,HdBoard,HdBoard,TA,TA,PConc,212,0,762,974,GasA,TA,Y,SBrkr,974,0,0,974,0,0,1,0,3,1,TA,5,Typ,0,0,0,Y,0,28,0,0,0,0,0,9,2007,Oth,Abnorml
1295,1296,20,RL,8400,Pave,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,5,5,1968,1968,Hip,CompShg,HdBoard,HdBoard,TA,TA,CBlock,1016,0,36,1052,GasA,Gd,Y,SBrkr,1052,0,0,1052,1,0,1,1,3,1,TA,5,Typ,0,1,288,Y,356,0,0,0,0,0,0,11,2006,WD,Normal
861,862,190,RL,11625,Pave,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,Norm,Norm,2fmCon,1Story,5,4,1965,1965,Hip,CompShg,Plywood,HdBoard,TA,TA,PConc,841,0,198,1039,GasA,Ex,Y,SBrkr,1039,0,0,1039,1,0,1,1,3,1,TA,6,Typ,0,2,504,Y,0,0,0,0,0,0,0,4,2010,WD,Normal


In [8]:
# custom class to apply log transformations

class logTransformer(BaseEstimator , TransformerMixin):

    def __init__(self, features):

        self.features = features

    def fit(self , X , y = None):

        return self

    def transform(self, X):

        X_transformed = X.copy()
        # perform the log transform on the data
        # make sure not to touch the entries with 0
        # check if all features exist
        for feature in self.features:

            if feature not in X_transformed.columns:
                raise ValueError(f'feature : {feature} not in dataframe')
        
        X_transformed[self.features] = np.log1p(X_transformed[self.features])

        return X_transformed


In [9]:
testing_set = logTransformer(features= ['GrLivArea' , 'TotalBsmtSF']).fit_transform(testing_set)
testing_set

,Id,MSSubClass,MSZoning,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,ExterQual,ExterCond,Foundation,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,GarageCars,GarageArea,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition
254,255,20,RL,8400,Pave,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,1Fam,1Story,5,6,1957,1957,Gable,CompShg,MetalSd,MetalSd,TA,Gd,CBlock,922,0,392,7.181592,GasA,TA,Y,SBrkr,1314,0,0,7.181592,1,0,1,0,3,1,TA,5,Typ,0,1,294,Y,250,0,0,0,0,0,0,6,2010,WD,Normal
1364,1365,160,FV,3180,Pave,Reg,Lvl,AllPub,Inside,Gtl,Somerst,Norm,Norm,TwnhsE,2Story,7,5,2005,2005,Gable,CompShg,MetalSd,MetalSd,Gd,TA,PConc,0,0,600,6.398595,GasA,Ex,Y,SBrkr,520,600,80,7.090910,0,0,2,1,2,1,Gd,4,Typ,0,2,480,Y,0,166,0,0,0,0,0,4,2006,WD,Abnorml
637,638,190,RM,6000,Pave,Reg,Lvl,AllPub,Inside,Gtl,OldTown,Norm,Norm,2fmCon,1.5Fin,5,4,1954,1954,Gable,CompShg,Wd Sdng,Wd Sdng,TA,TA,CBlock,0,0,811,6.699500,GasA,TA,Y,FuseA,811,576,0,7.235619,0,0,2,0,3,2,Gd,7,Typ,0,1,256,Y,0,0,0,0,0,0,0,11,2009,WD,Normal
974,975,70,RL,11414,Pave,IR1,Lvl,AllPub,Corner,Gtl,BrkSide,RRAn,Feedr,1Fam,2Story,7,8,1910,1993,Gable,CompShg,HdBoard,HdBoard,TA,Gd,BrkTil,0,0,728,6.591674,GasA,TA,N,SBrkr,1136,883,0,7.610853,0,0,1,0,3,1,Gd,8,Typ,0,2,532,Y,509,135,0,0,0,0,0,10,2009,WD,Normal
514,515,45,RL,10594,Pave,Reg,Lvl,AllPub,Inside,Gtl,Crawfor,Norm,Norm,1Fam,1.5Unf,5,5,1926,1950,Gable,CompShg,Wd Sdng,Wd Sdng,TA,TA,BrkTil,0,0,768,6.645091,Grav,Fa,N,SBrkr,789,0,0,6.672033,0,0,1,0,2,1,TA,5,Typ,0,1,200,Y,0,0,112,0,0,0,0,6,2007,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1096,1097,70,RM,6882,Pave,Reg,Lvl,AllPub,Inside,Gtl,IDOTRR,Norm,Norm,1Fam,2Story,6,7,1914,2006,Gable,CompShg,Wd Sdng,Wd Sdng,TA,TA,PConc,0,0,684,6.529419,GasA,TA,Y,SBrkr,773,582,0,7.212294,0,0,1,1,3,1,Gd,7,Typ,0,0,0,Y,136,0,115,0,0,0,0,3,2007,WD,Normal
1131,1132,20,RL,10712,Pave,Reg,Lvl,AllPub,Inside,Gtl,Mitchel,Norm,Norm,1Fam,1Story,5,5,1991,1992,Gable,CompShg,HdBoard,HdBoard,TA,TA,PConc,212,0,762,6.882437,GasA,TA,Y,SBrkr,974,0,0,6.882437,0,0,1,0,3,1,TA,5,Typ,0,0,0,Y,0,28,0,0,0,0,0,9,2007,Oth,Abnorml
1295,1296,20,RL,8400,Pave,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,5,5,1968,1968,Hip,CompShg,HdBoard,HdBoard,TA,TA,CBlock,1016,0,36,6.959399,GasA,Gd,Y,SBrkr,1052,0,0,6.959399,1,0,1,1,3,1,TA,5,Typ,0,1,288,Y,356,0,0,0,0,0,0,11,2006,WD,Normal
861,862,190,RL,11625,Pave,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,Norm,Norm,2fmCon,1Story,5,4,1965,1965,Hip,CompShg,Plywood,HdBoard,TA,TA,PConc,841,0,198,6.946976,GasA,Ex,Y,SBrkr,1039,0,0,6.946976,1,0,1,1,3,1,TA,6,Typ,0,2,504,Y,0,0,0,0,0,0,0,4,2010,WD,Normal


In [10]:
numerical_columns = [col for col in testing_set.columns if testing_set[col].dtype == 'int64' or testing_set[col].dtype == 'float64']
numerical_columns.remove('Id')

In [11]:
categorical_columns = [col for col in testing_set.columns if testing_set[col].dtype == 'O']

In [12]:
numeric_transformer = Pipeline(steps=[
    ('imputer' , SimpleImputer(strategy='median')) ,
    ('sacler' , StandardScaler())
]) 

categorical_tranformer = Pipeline(steps=[
    ('imputer' , SimpleImputer(strategy='most_frequent')),
    ('onehot' , OneHotEncoder(sparse_output=False , handle_unknown='ignore'))
])

In [13]:
preprocessor = ColumnTransformer(transformers=[
    ('num' , numeric_transformer , numerical_columns),
    ('cat' , categorical_tranformer , categorical_columns)
])

In [14]:
final_pipeline = Pipeline(steps=[
    ('dropColumns' , DropFeatures()),
    ('Transformation' , logTransformer(features= ['GrLivArea' , 'TotalBsmtSF'])),
    ('preprocessor' , preprocessor),
    ('regressor' , GradientBoostingRegressor(random_state=42 , n_estimators=500))
])

In [15]:
final_pipeline.fit(X_train , y_train)

,steps,"[('dropColumns', ...), ('Transformation', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,features_to_drop,"['LotFrontage', 'Alley', ...]"
,nan_threshold,1
,features,"['GrLivArea', 'TotalBsmtSF']"
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None


In [16]:
final_pipeline.score(X_test , y_test)

0.9152143889618511

In [17]:
# do it for the entire dataset
X_train = df.drop('SalePrice' , axis=1)
y_train = np.log(df['SalePrice'])
X_test = pd.read_csv('data/raw/test.csv')

In [18]:
final_pipeline.fit(X_train , y_train)

,steps,"[('dropColumns', ...), ('Transformation', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,features_to_drop,"['LotFrontage', 'Alley', ...]"
,nan_threshold,1
,features,"['GrLivArea', 'TotalBsmtSF']"
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None


In [19]:
y_pred = np.exp(final_pipeline.predict(X_test))

In [20]:
submission = X_test[['Id']]
submission['SalePrice'] = y_pred

/tmp/ipykernel_1109/668771410.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  submission['SalePrice'] = y_pred


In [21]:
submission

,Id,SalePrice
0,1461,121981.231930
1,1462,159365.130188
2,1463,186870.598514
3,1464,194892.270074
4,1465,183151.658243
...,...,...
1454,2915,78472.720856
1455,2916,81695.033797
1456,2917,148874.005510
1457,2918,117949.860746


In [22]:
submission.to_csv('data/submissions/GradientBosstingOnly.csv', index=False)

In [28]:
from sklearn.ensemble import RandomForestRegressor
final_pipeline = Pipeline(steps=[
    ('dropColumns' , DropFeatures()),
    ('Transformation' , logTransformer(features= ['GrLivArea' , 'TotalBsmtSF'])),
    ('preprocessor' , preprocessor),
    ('regressor' , RandomForestRegressor(random_state=42 , n_estimators=500))
])

final_pipeline.fit(X_train , y_train)
y_pred = np.exp(final_pipeline.predict(X_test))
submission = X_test[['Id']].copy()
submission['SalePrice'] = y_pred
submission.to_csv('data/submissions/RandomForestOnly.csv', index=False)

In [29]:
from xgboost import XGBRegressor
final_pipeline = Pipeline(steps=[
    ('dropColumns' , DropFeatures()),
    ('Transformation' , logTransformer(features= ['GrLivArea' , 'TotalBsmtSF'])),
    ('preprocessor' , preprocessor),
    ('regressor' , XGBRegressor(random_state=42 , n_estimators=1000))
])

final_pipeline.fit(X_train , y_train)
y_pred = np.exp(final_pipeline.predict(X_test))
submission = X_test[['Id']].copy()
submission['SalePrice'] = y_pred
submission.to_csv('data/submissions/XGBoostOnly.csv', index=False)

In [30]:
from sklearn.linear_model import LinearRegression
final_pipeline = Pipeline(steps=[
    ('dropColumns' , DropFeatures()),
    ('Transformation' , logTransformer(features= ['GrLivArea' , 'TotalBsmtSF'])),
    ('preprocessor' , preprocessor),
    ('regressor' , LinearRegression())
])

final_pipeline.fit(X_train , y_train)
y_pred = np.exp(final_pipeline.predict(X_test))
submission = X_test[['Id']].copy()
submission['SalePrice'] = y_pred
submission.to_csv('data/submissions/linearRegression.csv', index=False)